In [ ]:
import os

folders = os.listdir('/private/home/delong/workspace/subobjects-VLM/runs/clevr_caption')
for folder in folders:
    print(f"\"{folder.replace('SmolLM2-135M-Instruct-', '')}\",")

In [ ]:
import os, json
import matplotlib.pyplot as plt
import pprint
import numpy as np

embeddings = [
    # "dinov2_small(384px)",
    # "dinov2_small(768px)",
    # "dinov2_base(768px)",
    # "dinov2_large(768px)",
    # "zero_pixel(768px)",
    # "clip_vit_tinyclip(224px)",
    # "clip_vit_b_32(224px)",
    # "rgb_pixel(768px)",
    # "dinov2_small(64px)",
    # "dinov2_small(128px)",
    # "dinov2_small(192px)",
    # "vae(16px)",
    # "vae(24px)",
    "vae(32px)",
    "vae(48px)",
    "vae(64px)",
    "vae(128px)",
    "vae(256px)",
    "vae(384px)",
    "vae(512px)",
    "vae(768px)",
    "vae(1024px)",
]


all_root = '/private/home/delong/workspace/subobjects-VLM/runs/clevr_caption/SmolLM2-135M-Instruct-'

splits = ['val']


excluding = ['0107-1023-superpixel_slic(225t-384px)']

def get_loss_integration(trainer_state):
    sum = 0
    for step in trainer_state['log_history']:
        sum += step['loss']
    return sum / len(trainer_state['log_history'])

tokenizer_families = [
    'patch', 
    'superpixel', 
    'directsam'
]
all_results = []

for embedding in embeddings:
    root = all_root + embedding
    for tokenizer_family in tokenizer_families:
        # all_results[tokenizer_family] = []
        run_path = os.path.join(root, tokenizer_family)
        if os.path.exists(run_path):
            runs = os.listdir(run_path)
        for run in runs: 

            if tokenizer_family=='patch' and 'patch_10_per_side_random' not in run:
                continue
            if tokenizer_family=='superpixel' and 'superpixel_slic(100t' not in run:
                continue
            if tokenizer_family=='directsam' and 'directsam_tiny_sa1b_2ep@0_1(100t' not in run:
                continue


            path = os.path.join(root, tokenizer_family, run, 'runs')
            if run in excluding or not os.path.exists(path):
                continue
            files = os.listdir(path)

            files.sort()
            loss_integration = None
            for folder in files:
                if os.path.isdir(os.path.join(path, folder)) and folder.startswith('checkpoint-4080'):
                    trainer_state = os.path.join(path, folder, 'trainer_state.json')
                    trainer_state = json.load(open(trainer_state, 'r'))
                    loss_integration = get_loss_integration(trainer_state)
                    break
            if loss_integration is None:
                continue

            for file in files:
                # if file.startswith('eval_') and file.endswith('.json'):
                if file.startswith('vlm_eval_') and file.endswith('.json'):
                    with open(os.path.join(path, file), 'r') as f:
                        data = json.load(f)

            all_results.append({
                'embedding': embedding,
                'tokenizer_family': tokenizer_family,
                'run': run,
                # 'average_visual_tokens': data['average_visual_tokens'],
                # 'average_loss': data['average_loss'],
                # 'perplexity': np.exp(data['average_loss']).item(),
                'loss_integration': loss_integration,
            })
            loss_integration = 0

print(len(all_results))


In [ ]:
import pandas as pd
df = pd.DataFrame(all_results)
print(df)
df.to_csv('all_results.csv')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Make a copy if you don't want to overwrite your original df
# df_plot = df.copy()

# Rename the tokenizer families
df['tokenizer_family'] = df['tokenizer_family'].replace({
    'patch': 'Patch',
    'superpixel': 'Superpixel (SLIC)',
    'directsam': 'ECIT'
})

# Define a custom color palette
palette = {
    'Patch': 'gray',
    'Superpixel (SLIC)': 'purple',
    'ECIT': 'blue'
}

# Set a nice theme
sns.set_theme(style='whitegrid', context='talk')

plt.figure(figsize=(12, 6))  # Adjust size as needed

# Create the barplot
ax = sns.barplot(
    data=df,
    x='embedding',
    y='loss_integration',
    hue='tokenizer_family',
    palette=palette
)

# Set labels and title
ax.set_ylabel("Avg Training Loss")
ax.set_xlabel("Embedding")
ax.set_title("Loss Integration by Embedding and Tokenizer Family")

# Rotate x labels if needed
plt.xticks(rotation=0, ha="right")

plt.tight_layout()
plt.show()

In [19]:

# font_size = 25

# def visualize(all_results, y_key, y_label):
#     color_offset = 10

#     for split in splits:

#         plt.figure(figsize=(10, 10))
#         cmaps = ['Greys', 'Purples', 'Blues_r', 'Greens']

#         for tokenizer_family, results in all_results.items():
#             colors = plt.cm.get_cmap(cmaps.pop(0), len(results)+color_offset*2)

#             previous_x, previous_y = None, None

#             for i, result in enumerate(results):
#                 if result['evaluation_args']['split'] != split:
#                     continue

#                 if not result['evaluation_finished']:
#                     print(f"Skipping {result['evaluation_args']} because it didn't finish")
#                     # remove this result
#                     continue

#                 visual_tokenizer_config = result['checkpoint_args']['visual_tokenizer_config']
#                 tokenizer_name = visual_tokenizer_config.split('/')[-1].replace('.json', '')
#                 tokenizer_name = f"{tokenizer_name} ({result['checkpoint_args']['max_visual_tokens']})"

#                 plt.scatter(
#                     result['average_visual_tokens'], 
#                     result[y_key], 
#                     label=tokenizer_name, 
#                     color=colors(i+color_offset), 
#                     s=300,
#                     alpha=0.5
#                     )
#                 plt.scatter(
#                     result['average_visual_tokens'], 
#                     result[y_key], 
#                     label=tokenizer_name, 
#                     color=colors(i+color_offset), 
#                     s=100,
#                     )
#                 # plt.text(result['average_visual_tokens'], result[y_key], f'{tokenizer_name}', color=colors(i+color_offset))

#                 if previous_x is not None:
#                     plt.plot([previous_x, result['average_visual_tokens']], [previous_y, result[y_key]], color=colors(i+color_offset))

#                 previous_x, previous_y = result['average_visual_tokens'], result[y_key]

#         # plt.title(split)
#         # plt.legend(loc='lower right')
#         # make x axis log scale



#         plt.ylabel(y_label,  fontsize=font_size)

#         plt.grid(True, which='major', linestyle='--', alpha=0.8)
#         plt.grid(True, which='minor', linestyle=':', alpha=0.4)

#         plt.xscale('log')
#         plt.xlabel('# Tokens (log scale)',  fontsize=font_size)
#         ax = plt.gca()
#         xticks = [4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256]
#         # skip = [49, 81, 121, 144, 196, 225]
#         skip = [9, 49, 25, 81, 121, 144, 196, 225]
#         xticklabels = [str(x) if x not in skip else '' for x in xticks ]
#         ax.tick_params(axis='x', which='minor', bottom=False)  # Disable minor ticks


#         ax.set_xticks(xticks)
#         ax.set_xticklabels(xticklabels, fontsize=font_size*0.75)
        
#         # plt.xticks(fontsize=font_size*0.75)
#         plt.yticks(fontsize=font_size*0.75)
#         # plt.ylim(1, 2)
        
#         plt.show()

# print(root)
# visualize(all_results, y_key='loss_integration', y_label='Average Training Loss')
# visualize(all_results, y_key='average_perplexity', y_label='Validation Perplexity')